# Backtester za Trgovski Strategii so Tehnicki Indikatori

---

**Avtor:** [Vashe Ime]  
**Datum:** 2025  
**Predmet:** Programiranje / Finansiski Algoritmi

---

## Opis na Proektot

Ovoj proekt implementira **backtester** - sistem za testiranje na trgovski strategii na istorijski finansiski podatoci.

### Sto pravi ovoj proekt?
1. **Vcituva** istorijski CSV podatoci za finansiski instrumenti (akcii, kripto, itn.)
2. **Presmetuva** tehnicki indikatori: `RSI`, `SMA`, `EMA`, `MACD`
3. **Simulira** kupuvanje i prodavanje vrz osnova na pravila
4. **Vizuelizira** rezultatite so grafici
5. **Meri** performansite na sekoja strategija

### Strategii koi se testiraat:
- **Strategija 1:** SMA Crossover (Kratok vs. Dolg prosek)
- **Strategija 2:** RSI Oversold/Overbought

---

> **Zabeleska za pocetnici:** Sekoj del od kodot e detalno objasnет. Citaj gi komentarite!


## Del 1 - Uvoz na Biblioteki

Pred se, gi vcituvame site Python biblioteki koi ni trebaat:

| Biblioteka | Namena |
|---|---|
| `pandas` | Rabota so tabelarni podatoci |
| `numpy` | Matematicki presmetki |
| `matplotlib` | Crtanje na grafici |
| `warnings` | Skrivanje na nepotrebni predupreduvanja |

In [ ]:
# ============================================================
# UVOZ NA BIBLIOTEKI
# ============================================================

import pandas as pd          # Za rabota so DataFrame (tabeli)
import numpy as np           # Za matematicki operacii
import matplotlib.pyplot as plt  # Za crtanje na grafici
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Postavki za graficite
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print('Bibliotekite se uspesno vcitani!')
print(f'  pandas     : {pd.__version__}')
print(f'  numpy      : {np.__version__}')
print(f'  matplotlib : {plt.matplotlib.__version__}')

## Del 2 - Vcituvanje na Podatoci

### Opcija A: Generiranje na simulirani podatoci (preporacano za testiranje)
Ako nemas CSV fajl, kodot podolu avtomatski generira realisticni simulirani ceni so slucaen set (random walk).

### Opcija B: Vcituvanje na realen CSV
Otkomentira go delot so `pd.read_csv()` i vnesi go patot do tvojot CSV fajl.

**Format na CSV:**
```
Date,Open,High,Low,Close,Volume
2020-01-01,100.5,102.3,99.8,101.2,1500000
...
```

In [ ]:
# ============================================================
# VCITUVANJE / GENERIRANJE NA PODATOCI
# ============================================================

def generate_simulated_data(start_price=100.0, n_days=500, seed=42):
    """
    Generira simulirani OHLCV podatoci (Open, High, Low, Close, Volume).
    
    Parametri:
        start_price : pocetna cena
        n_days      : broj na denovi
        seed        : za reproducivilnost na slucajnite broevi
    """
    np.random.seed(seed)
    
    # Generiraме Close ceni so 'random walk' (slucaen cekor)
    # Ova e ednostaven model za simulacija na ceni na akcii
    daily_returns = np.random.normal(loc=0.0005, scale=0.015, size=n_days)
    close_prices  = start_price * np.cumprod(1 + daily_returns)
    
    # Generiraме Open, High, Low vrz osnova na Close
    noise        = lambda: np.random.uniform(0.005, 0.02, n_days)
    open_prices  = close_prices * (1 + np.random.normal(0, 0.005, n_days))
    high_prices  = np.maximum(close_prices, open_prices) * (1 + noise())
    low_prices   = np.minimum(close_prices, open_prices) * (1 - noise())
    volume       = np.random.randint(500_000, 5_000_000, n_days)
    
    # Pravime datumski indeks ('B' = rabotni denovi)
    dates = pd.date_range(start='2020-01-01', periods=n_days, freq='B')
    
    df = pd.DataFrame({
        'Open'  : open_prices,
        'High'  : high_prices,
        'Low'   : low_prices,
        'Close' : close_prices,
        'Volume': volume
    }, index=dates)
    df.index.name = 'Date'
    return df


# --- IZBERI EDEN OD SLEDNITE NACINI ---

# NACIN 1: Simulirani podatoci (aktivno)
df = generate_simulated_data(start_price=150.0, n_days=600)
print('Simulirani podatoci se generirani!')

# NACIN 2: Vcituvanje od CSV (otkomentira ako imas fajl)
# df = pd.read_csv('AAPL.csv', index_col='Date', parse_dates=True)
# df = df.sort_index()

# NACIN 3: Prezemanje od Yahoo Finance (otkomentira ako yfinance e instaliran)
# import yfinance as yf
# df = yf.download('AAPL', start='2020-01-01', end='2024-01-01')

# --- PRIKAZ NA OSNOVNI INFORMACII ---
print(f'\nInformacii za datasetot:')
print(f'  Broj na redovi : {len(df)}')
print(f'  Period         : {df.index[0].date()} do {df.index[-1].date()}')
print(f'  Koloni         : {list(df.columns)}')
print(f'\n  Pocetna cena   : ${df["Close"].iloc[0]:.2f}')
print(f'  Krajna cena    : ${df["Close"].iloc[-1]:.2f}')
print(f'  Min. cena      : ${df["Close"].min():.2f}')
print(f'  Maks. cena     : ${df["Close"].max():.2f}')
print()
df.head()

## Del 3 - Presmetuvanje na Tehnicki Indikatori

Tehnickit indikatori se matematicki formuli koi ni pomagaat da gi citame trendovite na pazarot.

### Indikatorite koi gi presmetuvame:

#### 1. SMA - Simple Moving Average (Ednostaven podvizen prosek)
Prosek na zatvorackit ceni vo odreden prozorec (n denovi).

#### 2. EMA - Exponential Moving Average
Slicen na SMA, no pogolema tezina im dava na ponovi vrednosti.

#### 3. RSI - Relative Strength Index
Meri dali instrumentot e **prekupen (>70)** ili **preprodaden (<30)**.

#### 4. MACD - Moving Average Convergence Divergence
Razlika megju 12-perioden i 26-perioden EMA. Dava signali za trend.

In [ ]:
# ============================================================
# PRESMETUVANJE NA TEHNICKI INDIKATORI
# ============================================================

def calculate_sma(series, window):
    """
    SMA - Simple Moving Average
    Primer: SMA(20) = prosek na posleednite 20 zatvoracki ceni
    """
    return series.rolling(window=window).mean()


def calculate_ema(series, window):
    """
    EMA - Exponential Moving Average
    adjust=False = klasicna EMA presmetka (rekurzivna formula)
    """
    return series.ewm(span=window, adjust=False).mean()


def calculate_rsi(series, window=14):
    """
    RSI - Relative Strength Index
    Vrednosti: 0-100
      > 70 = prekupen (mozna prodazba)
      < 30 = preprodaden (mozna kupuvacka)
    """
    # Presmetuvame dnevni promeni vo cenata
    delta = series.diff()
    
    # Gi razdeluvame pozitivnite i negativnite promeni
    gain = delta.clip(lower=0)   # Samo pozitivni (dobivki)
    loss = -delta.clip(upper=0)  # Samo negativni (zagubi)
    
    # Presmetuvame prosecna dobivka i zaguba
    avg_gain = gain.ewm(com=window - 1, min_periods=window).mean()
    avg_loss = loss.ewm(com=window - 1, min_periods=window).mean()
    
    # RS = prosecna dobivka / prosecna zaguba
    rs  = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def calculate_macd(series, fast=12, slow=26, signal=9):
    """
    MACD - Moving Average Convergence Divergence
    
    Vraka tri serii:
      - macd_line    : EMA(fast) - EMA(slow)
      - signal_line  : EMA(macd_line, signal)
      - histogram    : macd_line - signal_line
    """
    ema_fast    = calculate_ema(series, fast)
    ema_slow    = calculate_ema(series, slow)
    macd_line   = ema_fast - ema_slow
    signal_line = calculate_ema(macd_line, signal)
    histogram   = macd_line - signal_line
    return macd_line, signal_line, histogram


# ---- Gi presmetuvame site indikatori ----

# SMA
df['SMA_20']  = calculate_sma(df['Close'], 20)   # Kratok trend
df['SMA_50']  = calculate_sma(df['Close'], 50)   # Sreden trend
df['SMA_200'] = calculate_sma(df['Close'], 200)  # Dolg trend

# EMA
df['EMA_12'] = calculate_ema(df['Close'], 12)
df['EMA_26'] = calculate_ema(df['Close'], 26)

# RSI
df['RSI'] = calculate_rsi(df['Close'], window=14)

# MACD
df['MACD'], df['MACD_Signal'], df['MACD_Hist'] = calculate_macd(df['Close'])

print('Tehnickit indikatori se uspesno presmetani!')
indicator_cols = [c for c in df.columns if c not in ['Open','High','Low','Close','Volume']]
print(f'  Novi koloni: {indicator_cols}')
print()
cols_to_show = ['Close', 'SMA_20', 'SMA_50', 'EMA_12', 'RSI', 'MACD']
df[cols_to_show].tail()

## Del 4 - Vizuelizacija na Indikatorite

Grafikot sodrzi:
1. **Gore:** Cena + SMA linii
2. **Sredina:** RSI so granici na 30/70
3. **Dolu:** MACD histogram + signalna linija

In [ ]:
# ============================================================
# VIZUELIZACIJA NA INDIKATORITE
# ============================================================

def plot_indicators(df, title='Tehnicki Indikatori'):
    """
    Prikazuva tri paneli:
      1. Cena + SMA
      2. RSI
      3. MACD
    """
    fig = plt.figure(figsize=(16, 12))
    gs  = GridSpec(3, 1, figure=fig, height_ratios=[3, 1.5, 1.5], hspace=0.4)
    
    # --- Panel 1: Cena i Moving Averages ---
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(df.index, df['Close'],   label='Close cena', color='#2196F3', linewidth=1.5, alpha=0.9)
    ax1.plot(df.index, df['SMA_20'],  label='SMA 20',     color='#FF9800', linewidth=1.2, linestyle='--')
    ax1.plot(df.index, df['SMA_50'],  label='SMA 50',     color='#F44336', linewidth=1.2, linestyle='--')
    ax1.plot(df.index, df['SMA_200'], label='SMA 200',    color='#9C27B0', linewidth=1.5, linestyle='-.')
    ax1.set_title(title, fontsize=15, fontweight='bold', pad=12)
    ax1.set_ylabel('Cena ($)', fontsize=11)
    ax1.legend(loc='upper left', fontsize=9)
    ax1.tick_params(labelbottom=False)
    
    # --- Panel 2: RSI ---
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.plot(df.index, df['RSI'], label='RSI (14)', color='#00BCD4', linewidth=1.4)
    ax2.axhline(y=70, color='#F44336', linestyle='--', linewidth=1.0, alpha=0.7, label='Overbought (70)')
    ax2.axhline(y=30, color='#4CAF50', linestyle='--', linewidth=1.0, alpha=0.7, label='Oversold (30)')
    ax2.axhline(y=50, color='gray',    linestyle=':',  linewidth=0.8, alpha=0.5)
    ax2.fill_between(df.index, 70, 100, alpha=0.05, color='red')
    ax2.fill_between(df.index, 0,  30,  alpha=0.05, color='green')
    ax2.set_ylabel('RSI', fontsize=11)
    ax2.set_ylim(0, 100)
    ax2.legend(loc='upper left', fontsize=9)
    ax2.tick_params(labelbottom=False)
    
    # --- Panel 3: MACD ---
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    colors_hist = ['#4CAF50' if v >= 0 else '#F44336' for v in df['MACD_Hist']]
    ax3.bar(df.index, df['MACD_Hist'],   color=colors_hist, alpha=0.6, label='MACD Histogram', width=0.8)
    ax3.plot(df.index, df['MACD'],        label='MACD linija',    color='#2196F3', linewidth=1.2)
    ax3.plot(df.index, df['MACD_Signal'], label='Signalna linija', color='#FF9800', linewidth=1.2)
    ax3.axhline(y=0, color='white', linestyle='-', linewidth=0.5, alpha=0.3)
    ax3.set_ylabel('MACD', fontsize=11)
    ax3.set_xlabel('Datum', fontsize=11)
    ax3.legend(loc='upper left', fontsize=9)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig('indicators_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Grafikot e zacuvan kako 'indicators_chart.png'")


plot_indicators(df, title='Tehnicki Indikatori - Simulirani Podatoci')

## Del 5 - Backtesting Klasa

Ova e **srceto na proektot** - klasata `Backtester` koja:
- Cuva informacii za poceten kapital
- Simulira kupuvanje i prodavanje
- Gi sledi site trgovii (trejdovi)
- Presmetuva krajni metriki

### Logika na simulacijata:
```
Za sekoj den vo istorijata:
  Ako dobieme signal BUY  => kupuvame so celiot raspoloziv kapital
  Ako dobieme signal SELL => prodavame site akcii i cuvame gotovina
```

> **Poednostavuvanje:** Ne presmetuvame provizii/taksi (moze da se dodade podocna).

In [ ]:
# ============================================================
# BACKTESTER KLASA
# ============================================================

class Backtester:
    """
    Simulator na trgovski strategii vrz istorijski podatoci.
    
    Parametri:
        initial_capital : poceten kapital vo dolari
        commission      : provizija po trejd (0.001 = 0.1%)
    """
    
    def __init__(self, initial_capital=10_000.0, commission=0.001):
        self.initial_capital = initial_capital
        self.commission      = commission
    
    def run(self, df, signals, strategy_name='Strategija'):
        """
        Ja izvrssuva simulacijata.
        
        Parametri:
            df            : DataFrame so OHLCV podatoci
            signals       : Series so vrednosti 1 (BUY), -1 (SELL), 0 (HOLD)
            strategy_name : Naziv na strategijata
        
        Vraka:
            results_df : DataFrame so celata istorija na trgovija
            metrics    : Recnik so metriki za performansi
            trades_df  : DataFrame so site trejdovi
        """
        results  = df.copy()
        results['Signal'] = signals
        
        # Inicijaliziraме vrednosti
        cash     = self.initial_capital  # Gotovina
        shares   = 0.0                   # Broj na akcii
        position = 0                     # 0=nema, 1=kupeno
        
        portfolio_values = []
        trades           = []
        buy_signals_x    = []
        sell_signals_x   = []
        buy_signals_y    = []
        sell_signals_y   = []
        
        entry_price = 0.0
        entry_date  = None
        
        # ---- GLAVNA SIMULACISKA JAMKA ----
        for date, row in results.iterrows():
            price  = row['Close']
            signal = row['Signal']
            
            # Presmetuvame vrednostta na portfolioto
            portfolio_value = cash + shares * price
            portfolio_values.append(portfolio_value)
            
            # --- BUY signal i nemame pozicija ---
            if signal == 1 and position == 0:
                shares_to_buy   = (cash * (1 - self.commission)) / price
                cost            = shares_to_buy * price
                commission_paid = cost * self.commission
                shares          = shares_to_buy
                cash            = cash - cost - commission_paid
                position        = 1
                entry_price     = price
                entry_date      = date
                buy_signals_x.append(date)
                buy_signals_y.append(price)
            
            # --- SELL signal i imame pozicija ---
            elif signal == -1 and position == 1:
                revenue         = shares * price
                commission_paid = revenue * self.commission
                cash            = cash + revenue - commission_paid
                pnl             = revenue - (entry_price * shares) - commission_paid
                pnl_pct         = (price / entry_price - 1) * 100
                
                trades.append({
                    'Entry Date'  : entry_date,
                    'Exit Date'   : date,
                    'Entry Price' : round(entry_price, 2),
                    'Exit Price'  : round(price, 2),
                    'Shares'      : round(shares, 4),
                    'P&L ($)'     : round(pnl, 2),
                    'P&L (%)'     : round(pnl_pct, 2),
                    'Result'      : 'WIN' if pnl > 0 else 'LOSS'
                })
                sell_signals_x.append(date)
                sell_signals_y.append(price)
                shares   = 0.0
                position = 0
        
        results['Portfolio_Value'] = portfolio_values
        
        # ---- PRESMETUVANJE NA FINALNI METRIKI ----
        final_value  = results['Portfolio_Value'].iloc[-1]
        total_return = (final_value / self.initial_capital - 1) * 100
        bah_return   = (df['Close'].iloc[-1] / df['Close'].iloc[0] - 1) * 100
        
        trades_df = pd.DataFrame(trades) if trades else pd.DataFrame()
        n_trades  = len(trades)
        n_wins    = sum(1 for t in trades if t['P&L ($)'] > 0) if trades else 0
        n_losses  = n_trades - n_wins
        win_rate  = (n_wins / n_trades * 100) if n_trades > 0 else 0
        total_pnl = sum(t['P&L ($)'] for t in trades) if trades else 0
        avg_win   = np.mean([t['P&L ($)'] for t in trades if t['P&L ($)'] > 0]) if n_wins > 0 else 0
        avg_loss  = np.mean([t['P&L ($)'] for t in trades if t['P&L ($)'] <= 0]) if n_losses > 0 else 0
        
        rolling_max  = results['Portfolio_Value'].cummax()
        drawdown     = (results['Portfolio_Value'] - rolling_max) / rolling_max * 100
        max_drawdown = drawdown.min()
        
        metrics = {
            'strategy_name'   : strategy_name,
            'initial_capital' : self.initial_capital,
            'final_value'     : round(final_value, 2),
            'total_return_pct': round(total_return, 2),
            'buy_hold_return' : round(bah_return, 2),
            'n_trades'        : n_trades,
            'n_wins'          : n_wins,
            'n_losses'        : n_losses,
            'win_rate_pct'    : round(win_rate, 2),
            'total_pnl'       : round(total_pnl, 2),
            'avg_win'         : round(avg_win, 2),
            'avg_loss'        : round(avg_loss, 2),
            'max_drawdown_pct': round(max_drawdown, 2),
            'buy_signals'     : (buy_signals_x, buy_signals_y),
            'sell_signals'    : (sell_signals_x, sell_signals_y),
        }
        
        return results, metrics, trades_df


print('Backtester klasata e definirana i podgotvena za upotreba!')

## Del 6 - Strategija 1: SMA Crossover

### Ideja:
Ova e edna od **najpoznatite** strategii vo tehnickata analiza:

- **BUY (Kupi):** Koga SMA(20) ke go pomine SMA(50) nagore - "Golden Cross"
- **SELL (Prodaj):** Koga SMA(20) ke go pomine SMA(50) nadolu - "Death Cross"

```
SMA(20) > SMA(50)  =>  BUY
SMA(20) < SMA(50)  =>  SELL
```

**Intuicija:** Ako kratkoren prosek e nad dolgoren, pazarot e vo rascecki trend!

In [ ]:
# ============================================================
# STRATEGIJA 1 - SMA CROSSOVER
# ============================================================

def strategy_sma_crossover(df, fast=20, slow=50):
    """
    Generira trgovski signali vrz osnova na SMA Crossover.
    
    Vraka Series:
         1  = BUY  (kupi)
        -1  = SELL (prodaj)
         0  = HOLD (cekaj)
    """
    signals  = pd.Series(0, index=df.index)
    sma_fast = df[f'SMA_{fast}']
    sma_slow = df[f'SMA_{slow}']
    
    for i in range(1, len(df)):
        # GOLDEN CROSS: Kratkiot SMA go pominuva dolgiot nagore = BUY
        if (sma_fast.iloc[i] > sma_slow.iloc[i]) and \
           (sma_fast.iloc[i-1] <= sma_slow.iloc[i-1]):
            signals.iloc[i] = 1
        # DEATH CROSS: Kratkiot SMA go pominuva dolgiot nadolu = SELL
        elif (sma_fast.iloc[i] < sma_slow.iloc[i]) and \
             (sma_fast.iloc[i-1] >= sma_slow.iloc[i-1]):
            signals.iloc[i] = -1
    
    return signals


# Generiraме signali
signals_sma = strategy_sma_crossover(df, fast=20, slow=50)
print(f'Strategija 1 - SMA Crossover (SMA20 vs SMA50)')
print(f'  BUY signali  : {(signals_sma == 1).sum()}')
print(f'  SELL signali : {(signals_sma == -1).sum()}')

# Startuvame backtest
bt = Backtester(initial_capital=10_000, commission=0.001)
results_sma, metrics_sma, trades_sma = bt.run(df, signals_sma, 'SMA Crossover')

print(f'\nRezultati:')
print(f'  Poceten kapital    : ${metrics_sma["initial_capital"]:,.2f}')
print(f'  Krajna vrednost    : ${metrics_sma["final_value"]:,.2f}')
print(f'  Vkupen prinos      : {metrics_sma["total_return_pct"]:+.2f}%')
print(f'  Buy & Hold prinos  : {metrics_sma["buy_hold_return"]:+.2f}%')
print(f'  Broj na trejdovi   : {metrics_sma["n_trades"]}')
print(f'  Dobitni trejdovi   : {metrics_sma["n_wins"]} ({metrics_sma["win_rate_pct"]:.1f}%)')
print(f'  Gubitni trejdovi   : {metrics_sma["n_losses"]}')
print(f'  Maks. drawdown     : {metrics_sma["max_drawdown_pct"]:.2f}%')

## Del 7 - Strategija 2: RSI Oversold/Overbought

### Ideja:
RSI ni kazuva koga pazarot e pregolem panat (preprodaden) ili pregolem porasan (prekupen):

- **BUY:** Koga RSI < 30 => pazarot e preprodaden (ocekuvame odivanje nagore)
- **SELL:** Koga RSI > 70 => pazarot e prekupen (ocekuvame pad)

```
RSI < 30  =>  BUY  ("Kupi koga site prodavaat")
RSI > 70  =>  SELL ("Prodaj koga site kupuvaat")
```

**Intuicija:** Ova e **kontrarjanska** strategija - se dviziш sprotivno od tolpata!

In [ ]:
# ============================================================
# STRATEGIJA 2 - RSI OVERSOLD/OVERBOUGHT
# ============================================================

def strategy_rsi(df, oversold=30, overbought=70):
    """
    Generira signali vrz osnova na RSI pragovi.
    
    Parametri:
        oversold   : prag pod koj e signal za kupuvanje (default=30)
        overbought : prag nad koj e signal za prodavanje (default=70)
    """
    signals = pd.Series(0, index=df.index)
    rsi     = df['RSI']
    
    for i in range(1, len(df)):
        # BUY: RSI izlezel od preprodadenata zona (preminale 30 nagore)
        if (rsi.iloc[i] > oversold) and (rsi.iloc[i-1] <= oversold):
            signals.iloc[i] = 1
        # SELL: RSI izlezel od prekupenata zona (padnal pod 70)
        elif (rsi.iloc[i] < overbought) and (rsi.iloc[i-1] >= overbought):
            signals.iloc[i] = -1
    
    return signals


# Generiraме signali
signals_rsi = strategy_rsi(df, oversold=30, overbought=70)
print(f'Strategija 2 - RSI Oversold/Overbought')
print(f'  BUY signali  : {(signals_rsi == 1).sum()}')
print(f'  SELL signali : {(signals_rsi == -1).sum()}')

# Startuvame backtest
results_rsi, metrics_rsi, trades_rsi = bt.run(df, signals_rsi, 'RSI Strategija')

print(f'\nRezultati:')
print(f'  Poceten kapital    : ${metrics_rsi["initial_capital"]:,.2f}')
print(f'  Krajna vrednost    : ${metrics_rsi["final_value"]:,.2f}')
print(f'  Vkupen prinos      : {metrics_rsi["total_return_pct"]:+.2f}%')
print(f'  Buy & Hold prinos  : {metrics_rsi["buy_hold_return"]:+.2f}%')
print(f'  Broj na trejdovi   : {metrics_rsi["n_trades"]}')
print(f'  Dobitni trejdovi   : {metrics_rsi["n_wins"]} ({metrics_rsi["win_rate_pct"]:.1f}%)')
print(f'  Gubitni trejdovi   : {metrics_rsi["n_losses"]}')
print(f'  Maks. drawdown     : {metrics_rsi["max_drawdown_pct"]:.2f}%')

## Del 8 - Vizuelizacija na Trejdovite

Prikazuvame na grafik:
- Cenovnata linija
- **Zeleni triagolnici (^)** = BUY signali
- **Crveni triagolnici (v)** = SELL signali

In [ ]:
# ============================================================
# VIZUELIZACIJA NA TREJDOVITE - BUY/SELL SIGNALI
# ============================================================

def plot_trade_signals(df, metrics, title='Trgovski Signali'):
    """Prikazuva cenata so BUY (zeleno) i SELL (crveno) signali."""
    fig, ax = plt.subplots(figsize=(16, 7))
    
    ax.plot(df.index, df['Close'], label='Close cena', color='#2196F3',
            linewidth=1.4, alpha=0.85, zorder=1)
    
    if 'SMA_20' in df.columns:
        ax.plot(df.index, df['SMA_20'], label='SMA 20', color='#FF9800',
                linewidth=1, linestyle='--', alpha=0.7)
    if 'SMA_50' in df.columns:
        ax.plot(df.index, df['SMA_50'], label='SMA 50', color='#F44336',
                linewidth=1, linestyle='--', alpha=0.7)
    
    # BUY signali - zeleni triagolnici nagore
    buy_x, buy_y = metrics['buy_signals']
    ax.scatter(buy_x, buy_y, marker='^', color='#00C853', s=140,
               label=f'BUY ({len(buy_x)})', zorder=5, edgecolors='white', linewidth=0.8)
    
    # SELL signali - crveni triagolnici nadolu
    sell_x, sell_y = metrics['sell_signals']
    ax.scatter(sell_x, sell_y, marker='v', color='#FF1744', s=140,
               label=f'SELL ({len(sell_x)})', zorder=5, edgecolors='white', linewidth=0.8)
    
    ax.set_title(f"{title}\nPrinos: {metrics['total_return_pct']:+.2f}% | "
                 f"Trejdovi: {metrics['n_trades']} | "
                 f"Win rate: {metrics['win_rate_pct']:.1f}%",
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel('Cena ($)', fontsize=11)
    ax.set_xlabel('Datum', fontsize=11)
    ax.legend(loc='upper left', fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=30)
    plt.tight_layout()
    fname = f'signals_{title.replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Grafikot e zacuvan kako '{fname}'")


plot_trade_signals(df, metrics_sma, title='SMA Crossover Strategija')
plot_trade_signals(df, metrics_rsi, title='RSI Strategija')

## Del 9 - Equity Curve (Kriva na Kapitalot)

**Equity curve** pokazuva kako se menuvala vrednostta na naseto portfolio so tekot na vremeto.

- Ako linijata raste => strategijata e profitabilna
- Ako linijata opadnuva => gubime pari
- Sporeduvame so **Buy & Hold** (kolku bi zarabotile ako samo kupile i cekale)

Isto taka prikazuvame **Drawdown** - kolku % padnal kapitalot od negoviot vrv.

In [ ]:
# ============================================================
# EQUITY CURVE I DRAWDOWN
# ============================================================

def plot_equity_curves(df, results_list, metrics_list, initial_capital=10_000):
    """Prikazuva equity curves za povekje strategii + Buy & Hold sporedba."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                                    gridspec_kw={'height_ratios': [3, 1.5]})
    colors = ['#00E5FF', '#FF6D00', '#69F0AE', '#FF4081']
    
    # Buy & Hold (referenca)
    bah_values = initial_capital * (df['Close'] / df['Close'].iloc[0])
    bah_return = (bah_values.iloc[-1] / initial_capital - 1) * 100
    ax1.plot(df.index, bah_values,
             label=f'Buy & Hold ({bah_return:+.2f}%)',
             color='white', linewidth=1.5, linestyle=':', alpha=0.7)
    
    # Equity curves za sekoja strategija
    for i, (results, metrics) in enumerate(zip(results_list, metrics_list)):
        color = colors[i % len(colors)]
        name  = metrics['strategy_name']
        ret   = metrics['total_return_pct']
        ax1.plot(results.index, results['Portfolio_Value'],
                 label=f"{name} ({ret:+.2f}%)", color=color, linewidth=2.0)
    
    ax1.axhline(y=initial_capital, color='gray', linestyle='--',
                linewidth=0.8, alpha=0.5, label=f'Poceten kapital (${initial_capital:,})')
    ax1.set_title('Equity Curve - Sporedba na Strategii', fontsize=14,
                  fontweight='bold', pad=12)
    ax1.set_ylabel('Vrednost na Portfolio ($)', fontsize=11)
    ax1.legend(loc='upper left', fontsize=10)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # Drawdown
    for i, (results, metrics) in enumerate(zip(results_list, metrics_list)):
        color       = colors[i % len(colors)]
        name        = metrics['strategy_name']
        pv          = results['Portfolio_Value']
        rolling_max = pv.cummax()
        drawdown    = (pv - rolling_max) / rolling_max * 100
        ax2.fill_between(results.index, drawdown, 0, alpha=0.35, color=color, label=f'{name} DD')
        ax2.plot(results.index, drawdown, color=color, linewidth=1.0, alpha=0.8)
    
    ax2.set_title('Drawdown (%)', fontsize=12, fontweight='bold', pad=6)
    ax2.set_ylabel('Drawdown (%)', fontsize=10)
    ax2.set_xlabel('Datum', fontsize=11)
    ax2.legend(loc='lower left', fontsize=9)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.1f}%'))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig('equity_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Equity curve zacuvana kako 'equity_curves.png'")


plot_equity_curves(
    df,
    results_list=[results_sma, results_rsi],
    metrics_list=[metrics_sma, metrics_rsi],
    initial_capital=bt.initial_capital
)

## Del 10 - Lista na Trejdovi

Prikazuvame detalens tabela na site trgovii koi gi izvrsila sekoja strategija.

In [ ]:
# ============================================================
# LISTA NA TREJDOVI
# ============================================================

def print_trades(trades_df, strategy_name):
    """Ispisуva tabelata so trejdovi za edna strategija."""
    print('=' * 70)
    print(f'  LISTA NA TREJDOVI - {strategy_name.upper()}')
    print('=' * 70)
    if len(trades_df) == 0:
        print('  Nema zavrseni trejdovi.')
        return
    display_cols = ['Entry Date', 'Exit Date', 'Entry Price', 'Exit Price', 'P&L ($)', 'P&L (%)', 'Result']
    print(trades_df[display_cols].to_string(index=False))
    wins  = (trades_df['Result'] == 'WIN').sum()
    total = len(trades_df)
    print(f'\n  Vkupno: {total} trejdovi | WIN: {wins} | LOSS: {total-wins}')
    print(f'  Vkupen P&L: ${trades_df["P&L ($)"].sum():,.2f}')


print_trades(trades_sma, 'SMA Crossover')
print()
print_trades(trades_rsi, 'RSI Strategija')

## Del 11 - Finalen Izvestaj i Sporedba

Go prikazuvame **finalniot izvestaj** so sporedba na dvete strategii.

In [ ]:
# ============================================================
# FINALEN IZVESTAJ - SPOREDBA NA STRATEGII
# ============================================================

def print_final_report(metrics_list):
    """Ispetatуva tabela za sporedba na performansite."""
    print()
    print('=' * 65)
    print('  FINALEN IZVESTAJ - PERFORMANSI NA STRATEGII')
    print('=' * 65)
    
    rows = [
        ('Poceten kapital ($)',  lambda m: f"${m['initial_capital']:,.0f}"),
        ('Krajna vrednost ($)',  lambda m: f"${m['final_value']:,.2f}"),
        ('Vkupen prinos (%)',    lambda m: f"{m['total_return_pct']:+.2f}%"),
        ('Buy & Hold (%)',       lambda m: f"{m['buy_hold_return']:+.2f}%"),
        ('Broj na trejdovi',     lambda m: str(m['n_trades'])),
        ('Dobitni trejdovi',     lambda m: str(m['n_wins'])),
        ('Gubitni trejdovi',     lambda m: str(m['n_losses'])),
        ('Win Rate (%)',         lambda m: f"{m['win_rate_pct']:.1f}%"),
        ('Prosecna dobivka ($)', lambda m: f"${m['avg_win']:,.2f}"),
        ('Prosecna zaguba ($)',  lambda m: f"${m['avg_loss']:,.2f}"),
        ('Max Drawdown (%)',     lambda m: f"{m['max_drawdown_pct']:.2f}%"),
    ]
    
    # Header
    header = f"  {'Metrika':<28}"
    for m in metrics_list:
        header += f" {m['strategy_name']:<20}"
    print(header)
    print('-' * 65)
    
    for label, fn in rows:
        row = f"  {label:<28}"
        for m in metrics_list:
            row += f" {fn(m):<20}"
        print(row)
    
    print('=' * 65)
    best = max(metrics_list, key=lambda m: m['total_return_pct'])
    print(f"  Pobednik po prinos  : {best['strategy_name']} ({best['total_return_pct']:+.2f}%)")
    safest = min(metrics_list, key=lambda m: abs(m['max_drawdown_pct']))
    print(f"  Najbezbedna         : {safest['strategy_name']} (DD: {safest['max_drawdown_pct']:.2f}%)")
    print('=' * 65)


print_final_report([metrics_sma, metrics_rsi])

## Del 12 - Vizuelna Sporedba

In [ ]:
# ============================================================
# BARCHART SPOREDBA
# ============================================================

def plot_comparison_chart(metrics_list):
    """Bar-graf sporedba na kljucni metriki."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Sporedba na Strategii - Kljucni Metriki',
                 fontsize=14, fontweight='bold', y=1.02)
    
    names   = [m['strategy_name'] for m in metrics_list] + ['Buy & Hold']
    returns = [m['total_return_pct'] for m in metrics_list] + [metrics_list[0]['buy_hold_return']]
    colors  = ['#00E5FF', '#FF6D00', '#9E9E9E']
    
    # Graf 1: Vkupen prinos
    bars = axes[0].bar(names, returns, color=colors, edgecolor='white', linewidth=1.2)
    axes[0].set_title('Vkupen Prinos (%)', fontweight='bold')
    axes[0].set_ylabel('Prinos (%)')
    axes[0].axhline(y=0, color='white', linewidth=0.8, alpha=0.4)
    for bar, val in zip(bars, returns):
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                     f'{val:+.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Graf 2: Win Rate
    win_rates = [m['win_rate_pct'] for m in metrics_list]
    wr_colors = ['#69F0AE' if w >= 50 else '#FF5252' for w in win_rates]
    bars2 = axes[1].bar([m['strategy_name'] for m in metrics_list],
                        win_rates, color=wr_colors, edgecolor='white', linewidth=1.2)
    axes[1].axhline(y=50, color='yellow', linestyle='--', linewidth=1.2, alpha=0.7, label='50% linija')
    axes[1].set_title('Win Rate (%)', fontweight='bold')
    axes[1].set_ylabel('Win Rate (%)')
    axes[1].set_ylim(0, 100)
    axes[1].legend(fontsize=9)
    for bar, val in zip(bars2, win_rates):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Graf 3: Max Drawdown
    drawdowns = [abs(m['max_drawdown_pct']) for m in metrics_list]
    bars3 = axes[2].bar([m['strategy_name'] for m in metrics_list],
                        drawdowns, color=['#FF1744','#FF6D00'], edgecolor='white', linewidth=1.2)
    axes[2].set_title('Max Drawdown (%) - pomalo = podobro', fontweight='bold')
    axes[2].set_ylabel('Drawdown (%)')
    for bar, val in zip(bars3, drawdowns):
        axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('comparison_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Sporедbeniot grafik zacuvan kako 'comparison_chart.png'")


plot_comparison_chart([metrics_sma, metrics_rsi])

## Del 13 - Analiza i Zaklucocи

Vo ovoj del gi analizirame rezultatite i izvlekuvame zaklucocи.

In [ ]:
# ============================================================
# AVTOMATSKA ANALIZA NA REZULTATITE
# ============================================================

def auto_analysis(metrics_list):
    """Avtomatski generira analiza na rezultatite."""
    print('=' * 65)
    print('  ANALIZA NA REZULTATITE')
    print('=' * 65)
    
    bah = metrics_list[0]['buy_hold_return']
    print(f'\nBuy & Hold prinos (referenca): {bah:+.2f}%')
    print('  (Ova e prinos ako samo kupis i ne prodavas nicto)\n')
    
    for m in metrics_list:
        name = m['strategy_name']
        ret  = m['total_return_pct']
        diff = ret - bah
        print(f'Strategija: {name}')
        
        if ret > bah:
            print(f'  [+] Strategijata GO POBEDUVA Buy & Hold za {diff:+.2f}%')
        elif ret > 0:
            print(f'  [~] Profitabilna (+{ret:.2f}%) no NE go pobeduva Buy & Hold ({diff:.2f}%)')
        else:
            print(f'  [-] Strategijata e NEPROFITABILNA ({ret:.2f}%)')
        
        if m['n_trades'] > 0:
            print(f'  Broj trejdovi  : {m["n_trades"]}')
            print(f'  Win rate       : {m["win_rate_pct"]:.1f}% ({m["n_wins"]} WIN / {m["n_losses"]} LOSS)')
            print(f'  Max Drawdown   : {m["max_drawdown_pct"]:.2f}%')
            
            if m['avg_loss'] != 0:
                rr = abs(m['avg_win'] / m['avg_loss'])
                print(f'  Risk/Reward    : {rr:.2f}')
                if   rr > 1.5: print('  -> Dobar R:R odnos (>1.5)')
                elif rr > 1.0: print('  -> Prosecen R:R odnos')
                else:          print('  -> Los R:R odnos (<1.0) - zagubite se pogolemi od dobivkite')
        print()
    
    best   = max(metrics_list, key=lambda m: m['total_return_pct'])
    safest = min(metrics_list, key=lambda m: abs(m['max_drawdown_pct']))
    print('-' * 65)
    print(f'Najboljа strategija po prinos  : {best["strategy_name"]}')
    print(f'Najbezbedna (mal drawdown)     : {safest["strategy_name"]}')
    print()
    print('VAZNO: Rezultatite od backtest NE garantiraat idni performansi.')
    print('       Ova e samo obrazovna simulacija!')
    print('=' * 65)


auto_analysis([metrics_sma, metrics_rsi])

## Del 14 - Zakljucok na Proektot

---

### Sto naucivme?

Vo ovoj proekt implementiravme celos **backtesting sistem** za trgovski strategii:

1. **Tehnicki indikatori** - naucivme da presmetuvame SMA, EMA, RSI i MACD so pandas i numpy
2. **Simulacija na trgovija** - implementiravme realisticna simulacija so provizii i evidencija
3. **Strategii** - testiravme dva razlicni pristapi:
   - **SMA Crossover** - sledenje na trendot (trend-following)
   - **RSI Strategy** - kontrarianska strategija (contrarian)
4. **Metriki za performansi** - prinos, win rate, max drawdown, Risk/Reward
5. **Vizuelizacii** - price charts, signali, equity curve, drawdown

---

### Mozni podobruvanja:

| Podobruvanje | Opis |
|---|---|
| Stop Loss | Avtomatsko zatvoranje pri pad od X% |
| Position Sizing | Ne vlozuvas sekogash 100%, tuku delimicno |
| Portfolio | Istovremeno trguvaj so povekje akcii |
| Sharpe Ratio | Merka za rizik-prilagoden prinos |
| Walk-Forward Test | Pokrut nacin na testiranje |
| ML signali | Zamena na pravila so scikit-learn model |

---

> **Zadaca za studentite:** Probaj da gi promenish parametrite (SMA prozorec, RSI pragovi)
> i vidi kako toa vliguva na rezultatite!

---

### Referenci:
- [Investopedia - RSI](https://www.investopedia.com/terms/r/rsi.asp)
- [Investopedia - MACD](https://www.investopedia.com/terms/m/macd.asp)
- [pandas dokumentacija](https://pandas.pydata.org/docs/)

---
*Proektot e izraboten za obrazovni celi. Ne pretstavuva finansiski sovet.*